# 📚 MIDA507 -- Session 02
## Cycle de Vie des Données, Principes FAIR et Data Management Plan

| | |
|---|---|
| **Programme** | Master MIDA -- Université de Douala |
| **Session** | S02 -- Cycle DCC, FAIR, DMP, Data Lineage |
| **Durée estimée** | 90 à 120 minutes |
| **Prérequis** | Avoir exécuté le Notebook S01 |

### Ce que vous allez apprendre
1. Le cycle de vie des données selon le modèle DCC (7 étapes)
2. Ce qu'est un checksum SHA-256 et pourquoi il garantit l'intégrité
3. Les 4 principes FAIR et comment les évaluer sur un jeu africain
4. Rédiger la structure d'un Data Management Plan (DMP)
5. Documenter le data lineage (traçabilité des transformations)

### Livrable de la session
`MIDA507_S02_dmp.json` + `MIDA507_S02_rapport_fair.json`


---
## 📖 NOTION 1 -- Le Cycle de Vie des Données (modèle DCC)

### Définition en 3 lignes

Le **cycle de vie des données** décrit toutes les étapes par lesquelles passent les données, de leur création jusqu'à leur réutilisation ou destruction. Le modèle DCC (Digital Curation Centre) en identifie **7 étapes circulaires** qui se répètent à chaque mise à jour du jeu.

**Analogie IDA :** C'est l'équivalent du cycle de vie d'un document papier en archivistique -- création, instruction, versement, conservation, communication, élimination ou conservation définitive -- mais appliqué aux données numériques.

### Pourquoi c'est central pour l'IDA ?

L'IDA est le professionnel qui **supervise et documente** chaque étape. Sans cette supervision, les données se dégradent, se perdent ou deviennent inutilisables (problème courant dans les institutions africaines).

| Étape | Nom | Ce que fait l'IDA |
|---|---|---|
| 1 | **Planification** | Rédige le DMP, définit le schéma de métadonnées |
| 2 | **Collecte** | Vérifie la provenance, les droits, les formats |
| 3 | **Traitement** | Nettoie, normalise, documente les transformations |
| 4 | **Analyse** | Contextualise les résultats, identifie les biais |
| 5 | **Publication** | Choisit la licence, rédige la fiche DCAT |
| 6 | **Préservation** | Génère les checksums, choisit les formats pérennes |
| 7 | **Réutilisation** | Met à jour les métadonnées, collecte les retours |


In [ ]:
# ================================================================
# INSTALLATION ET CHARGEMENT DES OUTILS
# ================================================================
!pip install pandas seaborn matplotlib openpyxl --quiet

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import json, os, hashlib, random
from datetime import datetime, timedelta

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style="whitegrid")

print("✅ Outils installés et chargés.")
print("   Exécutez les cellules de haut en bas (Shift + Entrée).")


In [ ]:
# ================================================================
# RECHARGEMENT DU JEU DE DONNÉES BU-UNG (identique à la Session 01)
# ================================================================
# Ce bloc recrée le même jeu de données qu'en S01.
# La graine aléatoire 42 garantit des données identiques à chaque fois.
# Ne modifiez pas ce bloc -- exécutez-le simplement.
# ================================================================

random.seed(42)
NB = 5000

TYPES_DOCS  = ["Thèse et mémoire","Manuel universitaire","Revue scientifique",
               "Rapport de recherche","Ouvrage de référence","Document administratif"]
FILIERES    = ["Droit","Sciences économiques","Lettres modernes","Histoire",
               "Géographie","Médecine","Agronomie","Informatique"]
NIVEAUX     = ["Licence 1","Licence 2","Licence 3","Master 1","Master 2","Doctorat"]
REGIONS     = ["Adamaoua","Centre","Est","Extrême-Nord","Littoral","Nord","Ouest","Sud"]
LANGUES     = ["Français","Anglais","Bilingue FR/EN","Arabe","Autres langues africaines"]

d0 = datetime(2018,1,1)
dates = [(d0+timedelta(days=random.randint(0,365*5))).strftime("%Y-%m-%d") for _ in range(NB)]

emprunts = pd.DataFrame({
    "cote_document":   [f"UNG-{str(i+1).zfill(5)}" for i in range(NB)],
    "type_document":   random.choices(TYPES_DOCS,  weights=[0.28,0.30,0.15,0.10,0.10,0.07], k=NB),
    "date_emprunt":    dates,
    "duree_jours":     random.choices([3,7,7,14,14,14,21,30], k=NB),
    "code_usager":     [f"USR{str(random.randint(1,800)).zfill(4)}" for _ in range(NB)],
    "filiere":         random.choices(FILIERES, k=NB),
    "niveau_etude":    random.choices(NIVEAUX,  k=NB),
    "region_origine":  random.choices(REGIONS,  k=NB),
    "langue_document": random.choices(LANGUES,  weights=[0.62,0.22,0.10,0.04,0.02], k=NB),
})
emprunts["annee"] = pd.to_datetime(emprunts["date_emprunt"]).dt.year

print(f"✅ Jeu BU-UNG rechargé : {len(emprunts):,} emprunts, {len(emprunts.columns)} colonnes.")


In [ ]:
# ================================================================
# NOTION 1 EN PRATIQUE -- Journal du cycle de vie BU-UNG
# ================================================================
# On documente où en est notre jeu de données dans le cycle DCC.
# Chaque étape : statut actuel + actions réalisées + lacunes.
# ================================================================

# On crée une liste de dictionnaires (= liste de fiches structurées)
# Un dictionnaire Python = une fiche avec des champs nommés : {"champ": "valeur"}
journal_cycle_vie = [
    {
        "numero": 1,
        "etape": "Planification",
        "statut": "Partiellement réalisée",
        "actions_faites": ["Schéma de 10 colonnes défini", "Format CSV UTF-8 choisi"],
        "lacune_IDA": "Pas de DMP formel rédigé -- à corriger dans ce notebook"
    },
    {
        "numero": 2,
        "etape": "Collecte",
        "statut": "Réalisée (2018-2023)",
        "actions_faites": ["Saisie quotidienne dans le SIGB PMB", "Export CSV mensuel"],
        "lacune_IDA": "5% des champs 'filière' non renseignés"
    },
    {
        "numero": 3,
        "etape": "Traitement",
        "statut": "Réalisée (Session S01)",
        "actions_faites": ["Vérification qualité", "Ajout colonne 'annee'"],
        "lacune_IDA": "Pas de journal de transformations formalisé"
    },
    {
        "numero": 4,
        "etape": "Analyse",
        "statut": "Réalisée (Session S01 -- 7V)",
        "actions_faites": ["Analyse des 7V", "Tableau de bord produit"],
        "lacune_IDA": "Pas de contextualisation des biais identifiés"
    },
    {
        "numero": 5,
        "etape": "Publication",
        "statut": "En cours (Sessions S03-S06)",
        "actions_faites": [],
        "lacune_IDA": "Licence non choisie, fiche DCAT non rédigée, portail non sélectionné"
    },
    {
        "numero": 6,
        "etape": "Préservation",
        "statut": "Planifiée",
        "actions_faites": [],
        "lacune_IDA": "Aucun checksum calculé, format de préservation non défini"
    },
    {
        "numero": 7,
        "etape": "Réutilisation",
        "statut": "Planifiée",
        "actions_faites": [],
        "lacune_IDA": "Aucun mécanisme de retour utilisateur prévu"
    }
]

# -- Affichage du journal --
print("JOURNAL DU CYCLE DE VIE DCC -- Jeu BU-UNG Emprunts 2018-2023")
print("=" * 65)
print()

for etape in journal_cycle_vie:
    icone = "✅" if "Réalisée" in etape["statut"] else ("🔄" if "cours" in etape["statut"] else "⏳")
    print(f"  {icone}  Étape {etape['numero']} -- {etape['etape'].upper()}")
    print(f"     Statut : {etape['statut']}")
    if etape["lacune_IDA"]:
        print(f"     Lacune IDA : ⚠️  {etape['lacune_IDA']}")
    print()

# Sauvegarde du journal
with open("MIDA507_S02_journal_cycle_vie.json","w",encoding="utf-8") as f:
    json.dump(journal_cycle_vie, f, ensure_ascii=False, indent=2)
print("✅ Journal sauvegardé : MIDA507_S02_journal_cycle_vie.json")


---
## 📖 NOTION 2 -- Le Checksum SHA-256 : garantir l'intégrité des données

### Définition en 3 lignes

Un **checksum** (ou empreinte numérique) est un code unique calculé mathématiquement à partir du contenu d'un fichier. Si le fichier est modifié, même d'un seul caractère, le checksum change complètement. C'est une **preuve mathématique d'authenticité**.

**Analogie IDA :** C'est l'équivalent du numéro de collation d'un manuscrit (vérifier que tous les feuillets sont présents et non altérés), ou du scellé sur une boîte d'archives. En archivistique, on parle de **chaîne de custody numérique**.

### Pourquoi l'IDA doit le connaître ?

Avant de publier des données en open data ou de les archiver, l'IDA calcule l'empreinte du fichier et la conserve à part. À chaque vérification future, si le checksum recalculé est différent, le fichier a été altéré -- falsification prouvée.


In [ ]:
# ================================================================
# NOTION 2 EN PRATIQUE -- Calcul d'un checksum SHA-256
# ================================================================
# hashlib est la bibliothèque Python pour les fonctions de hachage.
# SHA-256 produit un code hexadécimal de 64 caractères.
# Cette fonction est irréversible : on ne peut pas retrouver le fichier
# original à partir de son empreinte.
# ================================================================

# -- Étape 1 : Convertir notre tableau en texte CSV --
# .to_csv() transforme le tableau en une chaîne de texte CSV
# encode("utf-8") convertit ce texte en octets (nécessaire pour hashlib)
contenu_csv = emprunts.to_csv(index=False).encode("utf-8")

print("CALCUL DU CHECKSUM SHA-256 -- Jeu BU-UNG")
print("=" * 55)
print()
print(f"  Taille du fichier CSV en mémoire : {len(contenu_csv):,} octets")
print()

# -- Étape 2 : Calculer l'empreinte SHA-256 --
# hashlib.sha256() crée un calculateur d'empreinte SHA-256
# .update() lui donne le contenu à analyser
# .hexdigest() retourne l'empreinte en hexadécimal (64 caractères)
calculateur = hashlib.sha256()
calculateur.update(contenu_csv)
empreinte_originale = calculateur.hexdigest()

print(f"  ✅ Empreinte SHA-256 du jeu ORIGINAL :")
print(f"     {empreinte_originale}")
print()

# -- Étape 3 : Simuler une modification frauduleuse --
# On modifie une seule valeur dans le tableau (durée d'emprunt ligne 0)
emprunts_modifie = emprunts.copy()     # .copy() crée une copie indépendante
emprunts_modifie.loc[0, "duree_jours"] = 999   # Modification frauduleuse !

# Calcul de l'empreinte du fichier modifié
contenu_modifie = emprunts_modifie.to_csv(index=False).encode("utf-8")
calc2 = hashlib.sha256()
calc2.update(contenu_modifie)
empreinte_modifiee = calc2.hexdigest()

print(f"  ⚠️  Empreinte SHA-256 après modification D'UNE SEULE valeur :")
print(f"     {empreinte_modifiee}")
print()

# -- Vérification : les empreintes sont-elles identiques ? --
if empreinte_originale == empreinte_modifiee:
    print("  ❌ PROBLÈME : les deux empreintes sont identiques -- la modification n'est pas détectée.")
else:
    print("  ✅ PREUVE : les deux empreintes sont DIFFÉRENTES.")
    print("     → Même un seul caractère modifié change complètement l'empreinte.")
    print("     → La falsification est détectée mathématiquement, instantanément.")

print()
print("  RÔLE IDA -- PRÉSERVATION :")
print("  → Calculez l'empreinte AVANT archivage et conservez-la dans un fichier séparé.")
print("  → À chaque vérification annuelle : recalculez et comparez.")
print("  → Si différent → le fichier a été altéré → restaurer la version archivée.")

# Sauvegarde du registre de checksums
registre = {
    "jeu_de_donnees": "BU-UNG Emprunts 2018-2023",
    "checksum_sha256": empreinte_originale,
    "date_calcul": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "nb_lignes": len(emprunts),
    "nb_colonnes": len(emprunts.columns)
}
with open("MIDA507_S02_registre_checksums.json","w",encoding="utf-8") as f:
    json.dump(registre, f, ensure_ascii=False, indent=2)
print()
print("✅ Registre de checksums sauvegardé : MIDA507_S02_registre_checksums.json")


---
## 📖 NOTION 3 -- Les Principes FAIR : l'étalon de qualité des données

### Définition en 3 lignes

Les **principes FAIR** (publiés en 2016) sont le standard international d'évaluation de la qualité des données ouvertes. Ils définissent 4 conditions pour qu'un jeu de données soit réellement réutilisable par d'autres institutions.

**Analogie IDA :** C'est l'équivalent d'un référentiel de catalogage international (comme ISBD ou MARC) -- mais pour les données numériques au lieu des documents bibliographiques.

### Les 4 principes FAIR

| Lettre | Principe | En une phrase | Analogie IDA |
|---|---|---|---|
| **F** | **Findable** (Trouvable) | Le jeu a un identifiant unique et ses métadonnées sont dans un catalogue | La cote unique d'un document dans un catalogue |
| **A** | **Accessible** | Le jeu est téléchargeable via un protocole ouvert (HTTPS), sans barrière | Le document est communicable en salle de lecture |
| **I** | **Interoperable** | Le jeu utilise des formats et vocabulaires standards | La notice MARC est compréhensible dans tous les SIGB |
| **R** | **Reusable** | Le jeu a une licence claire et sa provenance est documentée | Le droit de reproduire un document est clairement indiqué |

> 💡 **Un score FAIR de 100** = jeu parfaitement documenté, accessible et réutilisable.
> Notre jeu BU-UNG commence à **18/100** -- nous allons l'améliorer session par session.


In [ ]:
# ================================================================
# NOTION 3 EN PRATIQUE -- Évaluation FAIR du jeu BU-UNG
# ================================================================
# On crée un "profil FAIR" : une liste de critères avec leur note.
# Chaque critère vaut 0 (absent), 0.5 (partiel) ou 1 (complet).
# Le score final est sur 100 points.
# ================================================================

# -- Profil FAIR actuel de notre jeu (avant les sessions S03-S06) --
# None = non évalué, 0 = absent, 0.5 = partiel, 1 = complet

profil_fair = {

    # F -- FINDABLE (Trouvable) -- 4 critères
    "F1 -- Identifiant persistant (ARK, DOI...)": 0,      # Pas encore d'identifiant
    "F2 -- Titre descriptif complet": 0.5,                 # Titre court seulement
    "F3 -- Mots-clés (au moins 5)": 0,                    # Pas de mots-clés définis
    "F4 -- Présent dans un catalogue indexé": 0,           # Pas encore publié

    # A -- ACCESSIBLE -- 4 critères
    "A1 -- URL de téléchargement disponible": 0,           # Pas encore en ligne
    "A1a -- Protocole ouvert (HTTPS)": 0,                  # Pas encore publié
    "A1b -- Accès sans inscription": 0,                    # Pas encore publié
    "A2 -- Politique d'accès documentée": 0,               # Non définie

    # I -- INTEROPERABLE -- 3 critères
    "I1 -- Format ouvert (CSV, JSON, ODS)": 1,             # CSV = format ouvert ✅
    "I2 -- Standard de métadonnées (DCAT, Dublin Core)": 0, # Pas encore de fiche DCAT
    "I3 -- Vocabulaire contrôlé (AGROVOC, RAMEAU)": 0,    # Mots-clés libres seulement

    # R -- REUSABLE -- 4 critères
    "R1 -- Licence claire avec URI complète": 0,            # Pas de licence définie
    "R1a -- Provenance documentée": 1,                     # Journal cycle de vie ✅
    "R1b -- Journal des transformations": 0.5,             # Notebook S01 = partiel
    "R2 -- Contexte de collecte décrit": 0,                # Non décrit formellement
}

# -- Calcul du score FAIR sur 100 --
# sum() additionne toutes les valeurs, len() compte le nombre de critères
total_notes = sum(profil_fair.values())
nb_criteres = len(profil_fair)
score_fair = round(total_notes / nb_criteres * 100, 1)

# -- Affichage du bilan FAIR --
print("ÉVALUATION FAIR -- Jeu BU-UNG (état actuel)")
print("=" * 60)
print()
principe_actuel = ""
for critere, note in profil_fair.items():
    # Détection du principe (F, A, I, R) pour regrouper visuellement
    lettre = critere[0]
    if lettre != principe_actuel:
        principe_actuel = lettre
        noms = {"F":"FINDABLE (Trouvable)","A":"ACCESSIBLE","I":"INTEROPERABLE","R":"REUSABLE"}
        print(f"  --- {lettre} -- {noms[lettre]} ---")

    icone = "✅" if note == 1 else ("⚠️ " if note == 0.5 else "❌")
    barre = "█" * int(note * 10) + "░" * (10 - int(note * 10))
    print(f"  {icone} {critere:<50} : {note:.1f}  {barre}")

print()
print(f"  SCORE FAIR GLOBAL : {score_fair}/100")

if score_fair < 45:
    niveau = "❌ Insuffisant -- publication non recommandée en l'état"
elif score_fair < 65:
    niveau = "⚠️  Passable -- améliorations nécessaires avant publication"
elif score_fair < 85:
    niveau = "✅ Bon -- publiable avec quelques ajustements"
else:
    niveau = "🏆 Excellent -- jeu de référence"

print(f"  Niveau : {niveau}")
print()
print("  PLAN D'AMÉLIORATION FAIR :")
print("  Session S03 → Choisir la licence CC-BY (améliore R)")
print("  Session S04 → Rédiger la fiche DCAT (améliore F et I)")
print("  Session S06 → Publier sur CKAN (améliore A)")
print("  Session S07 → Documenter la provenance complète (améliore R)")
print()
print(f"  Cible après S03-S07 : ~87/100 (niveau Bon)")

# Sauvegarde
rapport_fair = {"score_actuel": score_fair, "profil": profil_fair,
                "date": datetime.now().strftime("%Y-%m-%d")}
with open("MIDA507_S02_rapport_fair.json","w",encoding="utf-8") as f:
    json.dump(rapport_fair, f, ensure_ascii=False, indent=2)
print()
print("✅ Rapport FAIR sauvegardé : MIDA507_S02_rapport_fair.json")


---
## 📖 NOTION 4 -- Le Data Management Plan (DMP)

### Définition en 3 lignes

Un **Data Management Plan** (DMP) est un document formel qui décrit comment un jeu de données sera collecté, documenté, stocké, partagé et préservé tout au long de son cycle de vie. Il est **exigé par les bailleurs de fonds** (AFD, Union Européenne, IDRC) comme condition de financement.

**Analogie IDA :** C'est l'équivalent d'un **plan de classement** combiné à une **politique d'archivage** -- il décrit à l'avance toutes les décisions de gestion documentaire avant que les données soient produites.

### Pourquoi l'IDA le rédige-t-il ?

Le DMP est un document **de gouvernance**, pas technique. Il engage l'institution sur ses pratiques de données. Seul un professionnel IDA maîtrise simultanément les aspects documentaires, juridiques et organisationnels nécessaires.

**Un DMP complet comprend 7 sections :**
1. Description des données (quoi)
2. Documentation et métadonnées (comment décrire)
3. Stockage et sécurité (où et comment protéger)
4. Aspects légaux et éthiques (droits, licences, données personnelles)
5. Partage et publication (qui peut accéder, sur quel portail)
6. Préservation à long terme (combien de temps, quel format)
7. Responsabilités et budget (qui fait quoi)


In [ ]:
# ================================================================
# NOTION 4 EN PRATIQUE -- Rédaction du DMP de la BU-UNG
# ================================================================
# On construit le DMP en Python comme un dictionnaire structuré.
# En situation réelle, on utiliserait DMPOnline (dmponline.dcc.ac.uk)
# ou l'Argos (e-infrastructure DMP française).
# ================================================================

dmp_bung = {
    "titre": "Data Management Plan -- Emprunts documentaires BU-UNG",
    "institution": "Bibliothèque Centrale -- Université de Ngaoundéré",
    "bailleur": "Carnegie Corporation of New York",
    "data_steward": "Data Steward IDA -- MIDA507",
    "version": "1.0",
    "date": datetime.now().strftime("%Y-%m-%d"),

    "section_1_description": {
        "titre": "1. Description des données",
        "types_donnees": ["Enregistrements d'emprunt (CSV tabulaire)", "Métadonnées DCAT (JSON)"],
        "volume_estime": "5 000 enregistrements par an -- environ 2 Mo par année",
        "format_principal": "CSV UTF-8",
        "source": "Export mensuel depuis le SIGB PMB de la Bibliothèque",
        "frequence_production": "Quotidienne (consolidation annuelle le 31 décembre)"
    },

    "section_2_metadonnees": {
        "titre": "2. Documentation et métadonnées",
        "standard_choisi": "DCAT-AP (W3C Data Catalog Vocabulary)",
        "champs_obligatoires": ["dct:identifier","dct:title","dct:description",
                                "dcat:keyword","dct:publisher","dct:license"],
        "outil_catalogue": "CKAN (à déployer en session S06)"
    },

    "section_3_stockage": {
        "titre": "3. Stockage et sécurité",
        "infrastructure": "Serveur SIGB PMB + backup Google Drive institutionnel",
        "frequence_backup": "Quotidienne automatique + mensuelle hors site",
        "acces_ecriture": "Data Steward IDA uniquement",
        "checksums": "SHA-256 calculé à chaque export -- registre conservé séparément"
    },

    "section_4_legal": {
        "titre": "4. Aspects légaux et éthiques",
        "licence_publication": "Creative Commons CC-BY 4.0",
        "donnees_personnelles": "Codes usagers présents -- pseudonymisation obligatoire avant publication",
        "loi_applicable": "Loi camerounaise 2010/012 sur la cybersécurité",
        "procedure_anonymisation": "À appliquer en session S07"
    },

    "section_5_partage": {
        "titre": "5. Partage et publication",
        "portail": "CKAN institutionnel Université de Ngaoundéré",
        "date_ouverture_prevue": "6 mois après début du projet",
        "format_diffusion": ["CSV UTF-8","JSON","API REST CKAN"],
        "embargo": "Aucun -- publication immédiate après pseudonymisation"
    },

    "section_6_preservation": {
        "titre": "6. Préservation à long terme",
        "formats_perennes": ["CSV UTF-8","JSON","PDF/A"],
        "entrepot": "Zenodo (CERN) pour les versions annuelles archivées",
        "duree_conservation": "10 ans après la dernière publication",
        "plan_migration": "Révision des formats tous les 5 ans"
    },

    "section_7_responsabilites": {
        "titre": "7. Responsabilités",
        "roles": {
            "Data Steward IDA": "Nettoyage, DCAT, publication, maintenance",
            "Direction Bibliothèque": "Validation des politiques, décisions d'accès",
            "DSI Université": "Infrastructure, backup, hébergement"
        },
        "budget_estime": "1 500 EUR/an (hébergement + formation)",
        "revision_dmp": "Tous les 6 mois ou lors de changements majeurs"
    }
}

# -- Affichage du DMP --
print("DATA MANAGEMENT PLAN -- Bibliothèque UNG")
print("=" * 60)
print(f"  Institution : {dmp_bung['institution']}")
print(f"  Bailleur    : {dmp_bung['bailleur']}")
print(f"  Responsable : {dmp_bung['data_steward']}")
print(f"  Version     : {dmp_bung['version']} ({dmp_bung['date']})")
print()

sections = ["section_1_description","section_2_metadonnees","section_3_stockage",
            "section_4_legal","section_5_partage","section_6_preservation","section_7_responsabilites"]

for section_key in sections:
    section = dmp_bung[section_key]
    print(f"  ✅ {section['titre']}")
    # On affiche le premier champ non-titre de chaque section
    for cle, valeur in section.items():
        if cle != "titre":
            if isinstance(valeur, list):
                print(f"     {cle} : {', '.join(str(v) for v in valeur[:2])}...")
            elif isinstance(valeur, dict):
                print(f"     {cle} : {list(valeur.keys())}")
            else:
                print(f"     {cle} : {str(valeur)[:60]}")
            break   # On affiche seulement le premier champ
    print()

# Sauvegarde
with open("MIDA507_S02_dmp.json","w",encoding="utf-8") as f:
    json.dump(dmp_bung, f, ensure_ascii=False, indent=2)
print("✅ DMP sauvegardé : MIDA507_S02_dmp.json")


---
## 📖 NOTION 5 -- Le Data Lineage : tracer chaque transformation

### Définition en 3 lignes

Le **data lineage** (ou lignage des données) est la documentation complète de toutes les transformations appliquées à un jeu de données, depuis sa création jusqu'à sa publication. Chaque modification est enregistrée : qui, quand, quoi, pourquoi.

**Analogie IDA :** C'est la **chaîne de custody** en archivistique -- le registre de tous les agents qui ont eu accès à un document et ce qu'ils ont fait. En numismatique documentaire, c'est l'historique complet de toutes les interventions sur un objet patrimonial.

### Pourquoi c'est crucial pour la recherche scientifique ?

Un chercheur qui cite votre jeu de données dans un article doit pouvoir vérifier que les données n'ont pas été altérées et comprendre toutes les transformations appliquées. Sans data lineage, la reproductibilité scientifique est impossible.


In [ ]:
# ================================================================
# NOTION 5 EN PRATIQUE -- Documenter le data lineage
# ================================================================
# On crée un journal des transformations.
# Chaque transformation = un enregistrement avec : qui, quand, quoi, pourquoi.
# ================================================================

journal_lineage = []   # Liste vide qui recevra nos enregistrements

# -- TRANSFORMATION 1 : Normalisation des noms de colonnes --
# Avant la transformation : on note l'état du tableau
colonnes_avant = list(emprunts.columns)

# Application de la transformation
emprunts_v2 = emprunts.copy()   # Copie du tableau original
emprunts_v2.columns = [col.lower().replace(" ","_") for col in emprunts_v2.columns]

colonnes_apres = list(emprunts_v2.columns)

# Enregistrement dans le journal
journal_lineage.append({
    "numero": 1,
    "date": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "responsable": "Data Steward IDA -- MIDA507",
    "outil": "Python pandas",
    "action": "Normalisation des noms de colonnes en minuscules sans espaces",
    "raison_IDA": "Conformité DCAT : les noms de colonnes ne doivent pas contenir d'espaces pour l'interopérabilité machine",
    "colonnes_avant": colonnes_avant,
    "colonnes_apres": colonnes_apres,
    "nb_lignes": len(emprunts_v2)
})

print("JOURNAL DE LINEAGE -- Transformation 1 enregistrée")
print("=" * 55)
t1 = journal_lineage[0]
print(f"  Date        : {t1['date']}")
print(f"  Responsable : {t1['responsable']}")
print(f"  Action      : {t1['action']}")
print(f"  Raison IDA  : {t1['raison_IDA']}")
print()
print(f"  Colonnes AVANT : {t1['colonnes_avant'][:3]}...")
print(f"  Colonnes APRES : {t1['colonnes_apres'][:3]}...")
print()

# -- TRANSFORMATION 2 : Calcul et ajout du checksum --
checksum = hashlib.sha256(emprunts_v2.to_csv(index=False).encode("utf-8")).hexdigest()

journal_lineage.append({
    "numero": 2,
    "date": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "responsable": "Data Steward IDA -- MIDA507",
    "outil": "Python hashlib.sha256",
    "action": "Calcul et enregistrement du checksum SHA-256 du jeu transformé",
    "raison_IDA": "Garantir l'intégrité du jeu à long terme -- toute altération future sera détectable",
    "checksum": checksum,
    "nb_lignes": len(emprunts_v2)
})

print("  ✅ Transformation 2 enregistrée : checksum calculé")
print(f"     SHA-256 : {checksum[:32]}...")
print()

# Sauvegarde du journal complet
with open("MIDA507_S02_journal_lineage.json","w",encoding="utf-8") as f:
    json.dump(journal_lineage, f, ensure_ascii=False, indent=2)
print("✅ Journal de lineage sauvegardé : MIDA507_S02_journal_lineage.json")
print()
print("  RÔLE IDA :")
print("  → Ce journal prouve la traçabilité de chaque modification.")
print("  → Un chercheur peut vérifier exactement quand et pourquoi")
print("    chaque transformation a été appliquée.")
print("  → En session S07, nous ajouterons la transformation d'anonymisation.")


---
## ✏️ EXERCICE -- Évaluez le FAIR d'un jeu de votre institution

**Consigne :** Choisissez un jeu de données réel ou réaliste de votre institution et estimez son score FAIR actuel.


In [ ]:
# ================================================================
# EXERCICE -- Score FAIR de mon institution
# ================================================================
# Modifiez UNIQUEMENT les valeurs None par 0, 0.5 ou 1.
# 0   = critère absent
# 0.5 = critère partiellement rempli
# 1   = critère complètement rempli
# ================================================================

MON_JEU = "[À COMPLÉTER : nom du jeu de données de votre institution]"

mon_profil_fair = {
    # F -- FINDABLE
    "F1 -- Identifiant persistant": None,
    "F2 -- Titre descriptif": None,
    "F3 -- Mots-clés (>=5)": None,
    "F4 -- Dans un catalogue": None,

    # A -- ACCESSIBLE
    "A1 -- URL de téléchargement": None,
    "A1a -- Protocole HTTPS": None,
    "A1b -- Accès sans inscription": None,
    "A2 -- Politique d'accès documentée": None,

    # I -- INTEROPERABLE
    "I1 -- Format ouvert (CSV, JSON...)": None,
    "I2 -- Standard de métadonnées": None,
    "I3 -- Vocabulaire contrôlé": None,

    # R -- REUSABLE
    "R1 -- Licence claire": None,
    "R1a -- Provenance documentée": None,
    "R1b -- Journal des transformations": None,
    "R2 -- Contexte de collecte décrit": None,
}

# -- Calcul automatique --
valides = {k:v for k,v in mon_profil_fair.items() if v is not None}
if valides:
    score = round(sum(valides.values()) / len(valides) * 100, 1)
    print(f"ÉVALUATION FAIR -- {MON_JEU}")
    print("=" * 55)
    for critere, note in mon_profil_fair.items():
        if note is not None:
            icone = "✅" if note==1 else ("⚠️" if note==0.5 else "❌")
            print(f"  {icone} {critere:<40} : {note}")
    print()
    print(f"  Score FAIR estimé : {score}/100")
    print(f"  ({len(valides)}/{len(mon_profil_fair)} critères évalués)")
else:
    print("[À COMPLÉTER]")
    print("Remplacez les 'None' par 0, 0.5 ou 1 pour calculer votre score FAIR.")
    print()
    print("Guide :")
    print("  0   = ce critère n'est PAS rempli pour votre jeu de données")
    print("  0.5 = ce critère est PARTIELLEMENT rempli")
    print("  1   = ce critère est COMPLÈTEMENT rempli")


---
## ✅ Bilan de la Session 02

| Compétence | Acquise |
|---|---|
| Expliquer le cycle de vie DCC en 7 étapes avec exemples IDA | ✅ |
| Calculer un checksum SHA-256 et expliquer son utilité archivistique | ✅ |
| Évaluer le score FAIR d'un jeu de données (14 critères) | ✅ |
| Structurer un DMP complet en 7 sections | ✅ |
| Documenter le data lineage de 2 transformations | ✅ |
| Évaluer le FAIR de mon propre jeu de données | 🟡 À compléter |

### Fichiers produits par cette session
- `MIDA507_S02_journal_cycle_vie.json` -- Les 7 étapes DCC de notre jeu
- `MIDA507_S02_registre_checksums.json` -- Empreinte SHA-256 de référence
- `MIDA507_S02_rapport_fair.json` -- Score FAIR actuel (18/100)
- `MIDA507_S02_dmp.json` -- Data Management Plan complet
- `MIDA507_S02_journal_lineage.json` -- Traçabilité des 2 transformations

*Notebook MIDA507 S02 -- Master MIDA -- Université de Douala*
